# Earthquake effect on house prices

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms

from read_cbs_data import *


In [2]:
CONFOUNDER_FOLDER_PATH = "./data/confounders/"

In [24]:
gdf_gemeenten = concatenate_cbs_gebieden(list(range(2010, 2026)), "gemeente_gegeneraliseerd")
gdf_provincies = concatenate_cbs_gebieden(list(range(2010, 2026)), "provincie_gegeneraliseerd")

gdf_joined = (
    join_gemeente_with_provincie(gdf_gemeenten, gdf_provincies)
    .rename({'statnaam__gemeente': "Regio's", 'jaar__gemeente': 'Perioden'}, axis = 1)
    )

df_prijsindex = read_prijsindex_data()

df_prijs_merged = gdf_joined.merge(df_prijsindex, left_on=["Regio's", 'Perioden'], right_on=['Gemeentenaam', 'Jaar'], how='inner')

df_confounder_merged = merge_confounder_data(df_prijs_merged)
df_confounder_merged = df_confounder_merged[df_confounder_merged['Perioden'] <= 2023]

df_aardbevingen = read_aardbevingen_data()
gdf_aardbevingen = transform_aardbevingen_data(df_aardbevingen)

In [27]:
df_confounder_merged.columns

Index(['Regio's', 'geometry', 'Perioden', 'statnaam__provincie',
       'Gemeentecode', 'Gemeentenaam', 'Index 2020=100',
       '95% betrouwbaarheidsmarge ondergrens',
       '95% betrouwbaarheidsmarge bovengrens',
       'Ontwikkeling t.o.v. voorgaande periode',
       'Ontwikkeling t.o.v. een jaar eerder', 'Jaar', 'Gehuwd', 'Gescheiden',
       'Ongehuwd', 'Verweduwd', 'Soort misdrijf',
       'Geregistreerde misdrijven/Totaal geregistreerde misdrijven (aantal)',
       'Gezondheid en welzijn/Huisartsenpraktijk/Afstand tot huisartsenpraktijk (km)',
       'Detailhandel/Winkels dagelijkse boodschappen/Afstand tot grote supermarkt (km)',
       'Horeca/Hotels en dergelijke/Aantal hotels e.d./Binnen 5 km (aantal)',
       'Horeca/Hotels en dergelijke/Aantal hotels e.d./Binnen 10 km (aantal)',
       'Horeca/Hotels en dergelijke/Aantal hotels e.d./Binnen 20 km (aantal)',
       'Onderwijs/Basisonderwijs/Afstand tot school (km)',
       'Werkgelegenheid: aantal banen/A-U alle economische

In [22]:
df_confounder_merged.groupby(["Regio's"])['Perioden'].size()

Regio's
Aa en Hunze        15
Aalsmeer           15
Aalten             15
Achtkarspelen      15
Alblasserdam       15
                   ..
Zundert            15
Zutphen            15
Zwartewaterland    15
Zwijndrecht        15
Zwolle             15
Name: Perioden, Length: 313, dtype: int64

In [23]:
df_confounder_merged['Perioden'].unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020,
       2021, 2022, 2023, 2024], dtype=int64)

In [31]:
gdf_aardbevingen['Date'] = gdf_aardbevingen['properties.time'].apply(lambda x: pd.to_datetime(x[:10]))
gdf_aardbevingen

,properties.time,properties.depth,properties.mag,properties.mag_type,properties.location,properties.event_type,geometry,Date
0,2025-12-21T18:54:21.8,3.000000,0.606156,MLn,Garrelsweer,induced or triggered event,POINT (6.775 53.292),2025-12-21
1,2025-12-16T22:33:47.299999,3.000000,0.477822,MLn,Woudbloem,induced or triggered event,POINT (6.745 53.226),2025-12-16
2,2025-12-16T12:18:23.199999,3.000000,1.989909,MLn,Woudbloem,induced or triggered event,POINT (6.745 53.228),2025-12-16
3,2025-12-16T12:10:00.146299,64.310890,3.745865,MLbes,Leeward Islands,earthquake,POINT (-63.34987 19.43309),2025-12-16
4,2025-12-16T10:02:00.651867,15.669783,2.230998,MLbes,Leeward Islands,earthquake,POINT (-62.81166 18.14548),2025-12-16
...,...,...,...,...,...,...,...,...
5696,1995-02-23T11:04:19.16,11.000000,2.300000,MLnq,Übach-Palenberg (Duitsland),earthquake,POINT (6.12333 50.92383),1995-02-23
5697,1995-02-01T00:31:32.0000,3.000000,1.200000,MLnq,Nieuw Annerveen,induced or triggered event,POINT (6.775 53.07867),1995-02-01
5698,1995-01-31T19:47:55.62,3.000000,2.040000,MLnq,Annen,induced or triggered event,POINT (6.72 53.06317),1995-01-31
5699,1995-01-24T09:38:39.19,3.000000,1.260000,MLnq,Delfzijl,induced or triggered event,POINT (6.89667 53.3155),1995-01-24


In [34]:
# Get the average magnitude and amount of earthquakes per year
gdf_aardbevingen.groupby(gdf_aardbevingen['Date'].dt.year)['properties.mag'].mean()
# gdf_aardbevingen.groupby(gdf_aardbevingen['Date'].dt.year).size()

Date
1995    1.944878
1996    1.703125
1997    1.649232
1998    1.937609
1999    1.710018
2000    1.777852
2001    1.405799
2002    1.666806
2003    1.564793
2004    1.592247
2005    1.828987
2006    1.542671
2007    1.761298
2008    1.668303
2009    1.566738
2010    1.559763
2011    1.570587
2012    1.240313
2013    1.433386
2014    1.334824
2015    1.110298
2016    0.912738
2017    1.019214
2018    1.191897
2019    1.049527
2020    1.204306
2021    1.020365
2022    1.071978
2023    1.677304
2024    2.154983
2025    2.201780
Name: properties.mag, dtype: float64

## Spatial join earthquake data on municipality

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from scipy.spatial import distance_matrix

# -------------------------------
# 1. Load your datasets
# -------------------------------

# Municipality polygons with yearly panel data
muni_gdf = gpd.read_file('municipalities.geojson')  # must have column 'municipality'
muni_panel = pd.read_csv('municipality_panel.csv')  # yearly data

# Earthquake points
eq_df = pd.read_csv('earthquakes.csv')  # columns: date, magnitude, latitude, longitude
eq_df['geometry'] = eq_df.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
eq_gdf = gpd.GeoDataFrame(eq_df, geometry='geometry', crs=muni_gdf.crs)

# -------------------------------
# 2. Compute municipality centroids
# -------------------------------
muni_gdf['centroid'] = muni_gdf.geometry.centroid
centroids = muni_gdf['centroid'].to_list()
muni_names = muni_gdf['municipality'].to_list()
n_muni = len(centroids)

# -------------------------------
# 3. Prepare earthquakes per year
# -------------------------------
eq_gdf['year'] = pd.to_datetime(eq_gdf['date']).dt.year
years = eq_gdf['year'].unique()
n_eq = len(eq_gdf)

# -------------------------------
# 4. Compute distance matrix (municipality centroid to all earthquakes)
# -------------------------------
# Extract coordinates
muni_coords = np.array([[c.x, c.y] for c in centroids])
eq_coords = np.array([[p.x, p.y] for p in eq_gdf.geometry])

# Euclidean distance in same CRS units (adjust CRS for meters/kilometers if needed)
dist_mat = distance_matrix(muni_coords, eq_coords)  # shape: (n_muni, n_eq)

# -------------------------------
# 5. Compute distance-weighted exposure
# -------------------------------
def exposure_weight(distance_km, magnitude, decay=50):
    """Exponential decay by distance (distance in km)."""
    return magnitude * np.exp(-distance_km / decay)

# Apply exposure weighting
mag_array = eq_gdf['magnitude'].values.reshape(1, -1)  # shape (1, n_eq)
exposure_mat = exposure_weight(dist_mat, mag_array)     # shape (n_muni, n_eq)

# -------------------------------
# 6. Aggregate exposure per municipality per year
# -------------------------------
muni_year_exposure = []

for i, muni in enumerate(muni_names):
    for year in years:
        mask = eq_gdf['year'].values == year
        yearly_exposure = exposure_mat[i, mask].sum()
        muni_year_exposure.append({
            'municipality': muni,
            'year': year,
            'distance_to_earthquake': yearly_exposure
        })

muni_year_df = pd.DataFrame(muni_year_exposure)

# -------------------------------
# 7. Merge with municipality panel data
# -------------------------------
panel_data = muni_panel.merge(muni_year_df, on=['municipality','year'], how='left')
panel_data['distance_to_earthquake'] = panel_data['distance_to_earthquake'].fillna(0)

# -------------------------------
# 8. OPTIONAL: compute spatial lag (neighbors) for spillovers
# -------------------------------
# Simple inverse distance weights (row-standardized)
inv_dist = 1 / np.where(dist_mat == 0, np.nan, dist_mat)
np.fill_diagonal(inv_dist, 0)
W = inv_dist / inv_dist.sum(axis=1, keepdims=True)

panel_data['lag_distance_to_earthquake'] = 0
for y in years:
    year_data = panel_data[panel_data['year'] == y]
    dist_vector = year_data['distance_to_earthquake'].values
    lag_vector = W @ dist_vector
    panel_data.loc[panel_data['year'] == y, 'lag_distance_to_earthquake'] = lag_vector

# -------------------------------
# 9. Set panel index for DiD / causal ML
# -------------------------------
panel_data = panel_data.set_index(['municipality','year'])

# -------------------------------
# 10. Ready for regression
# -------------------------------
print(panel_data.head())


## First standard panel DiD

For municapilty fixed effects and time FE

In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS

# Load your panel dataset
df = pd.read_csv('housing_earthquake_panel.csv')

# Set multi-index for panel (municipality × time)
df = df.set_index(['municipality', 'time'])

# Optional: demean continuous controls if you want (or include as covariates)
# Here we include gem_vermogen as a pre-treatment control
exog_vars = ['distance_to_earthquake', 'gem_vermogen']
exog = df[exog_vars]
exog = sm.add_constant(exog)

# Outcome variable
y = df['prijsindex_houses']

# Fit panel fixed effects model (municipality and time FE)
mod = PanelOLS(y, exog, entity_effects=True, time_effects=True)
res = mod.fit(cov_type='clustered', cluster_entity=True)

print(res.summary)

## Spatial panel DiD


Distance-weighted exposure of neighboring municipalities

Include spatial lag of treatment (W D_{it}) for spillovers

Optionally include spatial lag of outcome (W P_{it}) if housing markets are highly correlated

In [ ]:
import numpy as np
import libpysal
from scipy.spatial import distance_matrix

# Create coordinates array (municipality centroids)
coords = df.reset_index().drop_duplicates('municipality')[['x_coord','y_coord']].values

# Compute pairwise Euclidean distances
dist_mat = distance_matrix(coords, coords)

# Inverse distance weighting (replace zeros on diagonal to avoid division by zero)
inv_dist = 1 / np.where(dist_mat==0, np.nan, dist_mat)
np.fill_diagonal(inv_dist, 0)  # no self-weight

# Row-standardize the weights matrix
W = inv_dist / inv_dist.sum(axis=1, keepdims=True)

# Map municipality names to row indices
muni_names = df.reset_index()['municipality'].unique()
muni_map = {name: i for i, name in enumerate(muni_names)}

# Create spatially lagged treatment
df_reset = df.reset_index()
df_reset['lag_distance_to_earthquake'] = df_reset.apply(
    lambda row: np.dot(W[muni_map[row['municipality']], :], 
                       df_reset[df_reset['time']==row['time']]['distance_to_earthquake'].values),
    axis=1
)

# Re-set index
df_spatial = df_reset.set_index(['municipality','time'])

# Exogenous variables now include direct + spatial lag
exog_vars = ['distance_to_earthquake', 'lag_distance_to_earthquake', 'gem_vermogen']
exog = sm.add_constant(df_spatial[exog_vars])
y = df_spatial['prijsindex_houses']

# Fit PanelOLS with FE
mod_spatial = PanelOLS(y, exog, entity_effects=True, time_effects=True)
res_spatial = mod_spatial.fit(cov_type='clustered', cluster_entity=True)

print(res_spatial.summary)